# 184 — Banking / ATM / Geospatial
Trains `models/184_model.pkl`. Checklist: plan.md §3. Heads: risk/alert (synthetic `is_suspicious`) and predicted withdrawal-city (atm_withdrawal_links). External `synthetic_financial_fraud` is empty here, so 184's own labeled synthetic links + live OSM ATM counts are the signal.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path(os.getcwd()).resolve()
while not (ROOT / "lib").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
from pathlib import Path

import yaml

cfg_path = ROOT / "config" / "run_config.yaml"
with open(cfg_path) as fh:
    RUN = yaml.safe_load(fh)
print(
    "run config:", {k: RUN[k] for k in ("active_models", "input_source", "eval_mode")}
)
TH = RUN.get("thresholds", {})

In [ ]:
import lib.artifacts as art
import lib.io_utils as io

io.ensure_dirs()

## 3.1 Data ingestion

In [ ]:
syn = io.load_184_synthetic()
txns = (syn.get("bank_transactions") or {}).get("transactions", [])
links = (syn.get("atm_withdrawal_links") or {}).get("links", [])
print("bank transactions:", len(txns), "| atm withdrawal links:", len(links))
ref = io.load_184_reference()
print("reference cities:", len(ref.get("cities", []) or ref.get("cities", [])), end=" ")
osm = io.load_184_external_osm()
print("| live OSM city extracts:", list(osm.keys()))
cfpb184 = io.load_184_cfpb(n=50_000)
print("184 CFPB rows:", len(cfpb184))

## 3.2 / 3.3 Feature engineering & labels
Transaction-graph + amount + route features; withdrawal-city label derived by joining scenario_id → ATM → city; risk label from `is_suspicious`.

## 3.4 / 3.5 Modeling & evaluation (hit-rate@k for withdrawal city)

In [ ]:
import lib.model_184 as m184

model = m184.Model184().fit()
print("=== 184 metrics (TEST / held-out) ===")
for head, m in model.metrics.items():
    print(head, {k: round(v, 4) if isinstance(v, float) else v for k, v in m.items()})
print("--- TRAIN (fit) ---")
for head, m in model.train_metrics.items():
    print(head, {k: round(v, 4) if isinstance(v, float) else v for k, v in m.items()})

## 3.6 Output — serialize + smoke test

In [ ]:
from lib import schema

art.save_model(
    model,
    io.MODELS_DIR / "184_model.pkl",
    provenance={"model": "184", "metrics": model.metrics},
)
print("saved", io.MODELS_DIR / "184_model.pkl")
rec = txns[0]
payload = model.predict(rec, threshold=TH.get("alert", 0.7))
assert schema.is_valid(payload)
cid = (rec.get("bank_transaction_data", {}) or {}).get("transaction_id", "txn")
out_path = io.write_out(184, payload, "smoke", case_id=cid)
print("smoke-test wrote:", out_path.name, "| validated: OK")